# World Maps — imagination as a physical *process*: video generators inside analysis-by-synthesis (Colab)

Notebook 02 lets an image generator *edit* a candidate into a draped version. This notebook asks whether *simulating the
process* does better: an image-to-video model watches a cloth **fall onto** each candidate (or get **pulled off** the
target), and the last frame is the imagined end state. Same trials, same human data, same scoring as notebook 02, so
the two routes (appearance edit vs physical process) are compared directly against humans and PbAS.

| Stage | What |
|---|---|
| Video imagination | LTX-Video 2B (default, ≈ 10–20 s per clip at 512 px; the 13B "dev" checkpoints are not used) or Wan 2.2 TI2V-5B (≈ 1–2 min, optional) image-to-video; K clips per image; last frame saved; a few mp4s kept for inspection |
| Encoders | the models in `MODELS` embed the stimuli and the imagined frames |
| Analysis | humans, PbAS reference, raw readouts, de-draping, image-imagination (if present from notebook 02 on Drive) and video-imagination readouts with the sequential-sampling RT model |

**Runtime:** A100. K = 2 with LTX 2B at 512 px ≈ 1.5–2.5 h for 480 + 240 clips (expect about 12 GB of GPU memory);
set `IMAGINE_LIMIT` for a partial run (the analysis needs all 120 trials to score, partial runs are for inspecting quality). The last cell downloads
`world_maps_cloth_video_results.zip`. Nothing is pushed to GitHub.


In [ ]:
#@title 1. Configuration
MODELS = ["dinoB", "dinoL", "siglip2", "clipL"]  #@param {type:"raw"}
VIDEO_GENERATOR = "ltx"     #@param ["ltx", "wan"]
K_SAMPLES = 2               #@param {type:"integer"}
VIDEO_SIZE = 512            #@param {type:"integer"}   # multiple of 32
N_FRAMES = 49               #@param {type:"integer"}   # 8k+1 for LTX, 4k+1 for Wan
DIRECTIONS = ["drape", "undrape"]  #@param {type:"raw"}
IMAGINE_LIMIT = None        #@param {type:"raw"}       # e.g. 12 trials to inspect quality first
SMOKE_TEST_FIRST = True     #@param {type:"boolean"}
LOAD_FROM_DRIVE = False     #@param {type:"boolean"}   # reuse results/cloth/features and imagined images saved by notebook 02

import os, torch
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch to a GPU runtime)'}  {gpu_gb:.0f} GB")
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok: os.environ["HF_TOKEN"] = tok; print("HF_TOKEN loaded from Colab secrets")
except Exception:
    pass


In [ ]:
#@title 2. Code and dependencies (about 2 min)
%cd /content
import os
if not os.path.exists("world-maps"):
    !git clone -q https://github.com/physicalreasoning/world-maps.git
%cd /content/world-maps
!git pull -q; git rev-parse --short HEAD
!pip install -q "transformers>=5.0" accelerate "diffusers>=0.33" imageio imageio-ffmpeg remotezip matplotlib pillow opencv-python-headless scikit-learn scipy pandas safetensors
import torch, transformers, diffusers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| diffusers", diffusers.__version__)
os.environ["PYTHONPATH"] = "/content/world-maps/src"
os.makedirs("results/cloth", exist_ok=True)
if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r /content/drive/MyDrive/world_maps_cloth/features results/cloth/ 2>/dev/null; cp -r /content/drive/MyDrive/world_maps_cloth/imagined results/cloth/ 2>/dev/null; ls results/cloth
!PYTHONPATH=src python -c "from world_maps import cloth; cloth.fetch_assets(); print('assets ready')"


In [ ]:
%%writefile /content/world-maps/src/world_maps/cloth.py
"""Shape under cloth (Yildirim, Siegel, Soltani, Ray Chaudhuri & Tenenbaum 2024, Nat Hum Behav).

Assets (public, fetched at run time):
  stimuli   github.com/CNCLgithub/intuitive-physics-3d-shape-perception-stimuli  (720 PNG, 224 x 224)
  behaviour github.com/CNCLgithub/PbAS-model-human-comparisons  (20,880 trials, 174 workers; PbAS runs)

Task: 2AFC match-to-sample. Trial t in 0..119 has a target (cloth-draped object in the occluded
condition, the bare object in another pose in the unoccluded condition), the matching object and a
distractor, both bare and in random poses. Trials come in blocks of six; even blocks are same-category
(harder), odd blocks different-category (easier), as in the lab's analysis code.

Human processing follows the lab's scripts: workers with overall accuracy <= 71/120 are excluded;
trial accuracy is the mean of ``correct`` per (condition, presentation time, trial); response times drop
|z| >= 3 within a worker-condition before averaging; participant bootstraps resample workers.
"""

from __future__ import annotations

import subprocess
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from .paths import DATA

CLOTH = DATA / "cloth"
STIM = CLOTH / "stimuli"
BEH = CLOTH / "behaviour"
STIM_REPO = "https://github.com/CNCLgithub/intuitive-physics-3d-shape-perception-stimuli"
BEH_REPO = "https://github.com/CNCLgithub/PbAS-model-human-comparisons"
CATEGORIES = [
    "airplane",
    "bicycle",
    "bus",
    "car",
    "chair",
    "guitar",
    "motorcycle",
    "pistol",
    "rifle",
    "table",
]
TIMES = {"1000": "1s", "2000": "2s", "0": "unlimited"}
CONDITIONS = {0: "unoccluded", 1: "occluded"}
PBAS_ITER = {"1s": 48, "2s": 80, "unlimited": 110}  # best-matching iterations reported in the paper
PIXEL_PBAS_ITER = {"1s": 84, "2s": 116, "unlimited": 140}
ACC_THRESHOLD = 71.0 / 120


def fetch_assets() -> None:
    CLOTH.mkdir(parents=True, exist_ok=True)
    for repo, dest in [(STIM_REPO, STIM), (BEH_REPO, BEH)]:
        if not dest.exists():
            subprocess.run(["git", "clone", "-q", "--depth", "1", repo, str(dest)], check=True)


@dataclass
class Trial:
    t: int
    same_category: bool
    block: int

    def images(self, condition: str) -> dict[str, Path]:
        """Paths for target, match and distractor in one occlusion condition."""
        if condition == "occluded":
            d = STIM / "unoccluded-occluded"
            return {
                "target": d / f"trial_{self.t}_gtCloth.png",
                "match": d / f"trial_{self.t}_gtNoCloth.png",
                "distractor": d / f"trial_{self.t}_disNoCloth0.png",
            }
        d = STIM / "unoccluded-unoccluded"
        return {
            "target": d / f"trial_{self.t}_gtTargetNoCloth.png",
            "match": d / f"trial_{self.t}_gtNoCloth.png",
            "distractor": d / f"trial_{self.t}_disNoCloth0.png",
        }


def trials() -> list[Trial]:
    return [Trial(t=t, same_category=((t // 6) % 2 == 0), block=t // 6) for t in range(120)]


def image_index() -> pd.DataFrame:
    """All 720 stimulus images with (condition, trial, role, path)."""
    rows = []
    for tr in trials():
        for cond in CONDITIONS.values():
            for role, p in tr.images(cond).items():
                rows.append(
                    {
                        "condition": cond,
                        "trial": tr.t,
                        "role": role,
                        "path": str(p),
                        "same_category": tr.same_category,
                    }
                )
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- behaviour


def load_behaviour() -> pd.DataFrame:
    df = pd.read_csv(BEH / "processed_data_anonymous.csv")
    df.columns = ["workerid", "condition", "stim_present", "stimulus", "response", "correct", "rt"]
    df["time"] = df["stim_present"].astype(str).map(TIMES)
    df["cond"] = df["condition"].map(CONDITIONS)
    acc = df.groupby("workerid")["correct"].mean()
    keep = acc[acc > ACC_THRESHOLD].index
    df = df[df.workerid.isin(keep)].copy()
    df["same_category"] = (df.stimulus // 6) % 2 == 0
    return df


def human_trial_table(df: pd.DataFrame) -> pd.DataFrame:
    """Per (cond, time, trial): accuracy, n, mean RT after the |z| < 3 filter, mean log RT."""
    out = (
        df.groupby(["cond", "time", "stimulus"])
        .agg(acc=("correct", "mean"), n=("correct", "size"))
        .reset_index()
    )
    rts = []
    for (_w, _c, _t), g in df.groupby(["workerid", "cond", "time"]):
        z = (g.rt - g.rt.mean()) / (g.rt.std() + 1e-9)
        rts.append(g[np.abs(z) < 3])
    rt = (
        pd.concat(rts)
        .groupby(["cond", "time", "stimulus"])
        .agg(rt=("rt", "mean"), log_rt=("rt", lambda x: np.log(x).mean()))
        .reset_index()
    )
    out = out.merge(rt, on=["cond", "time", "stimulus"], how="left")
    out["same_category"] = (out.stimulus // 6) % 2 == 0
    return out.rename(columns={"stimulus": "trial"})


def participant_bootstrap(
    df: pd.DataFrame, cond: str, time: str, n_boot: int = 2000, seed: int = 0
) -> np.ndarray:
    """Per-trial accuracy under participant resampling: array (n_boot, 120)."""
    sub = df[(df.cond == cond) & (df.time == time)]
    workers = sub.workerid.unique()
    mats = {
        w: g.set_index("stimulus")["correct"].reindex(range(120)).values for w, g in sub.groupby("workerid")
    }
    M = np.array([mats[w] for w in workers], float)  # workers x 120 (NaN where missing)
    rng = np.random.default_rng(seed)
    out = np.zeros((n_boot, 120))
    for b in range(n_boot):
        idx = rng.integers(0, len(workers), len(workers))
        out[b] = np.nanmean(M[idx], 0)
    return out


def split_half_ceiling(
    df: pd.DataFrame, cond: str, time: str, which=None, n_splits: int = 1000, seed: int = 0
) -> dict:
    """Bootstrapped split-half correlation of per-trial accuracy across participants (the lab's method)."""
    sub = df[(df.cond == cond) & (df.time == time)]
    workers = sub.workerid.unique()
    mats = {
        w: g.set_index("stimulus")["correct"].reindex(range(120)).values for w, g in sub.groupby("workerid")
    }
    M = np.array([mats[w] for w in workers], float)
    sel = np.arange(120) if which is None else np.array(which)
    rng = np.random.default_rng(seed)
    rs = []
    for _ in range(n_splits):
        perm = rng.permutation(len(workers))
        a, b = perm[: len(perm) // 2], perm[len(perm) // 2 :]
        x, y = np.nanmean(M[a], 0)[sel], np.nanmean(M[b], 0)[sel]
        if x.std() > 0 and y.std() > 0:
            rs.append(np.corrcoef(x, y)[0, 1])
    rs = np.array(rs)
    return {
        "mean": float(rs.mean()),
        "ci": [float(np.percentile(rs, 2.5)), float(np.percentile(rs, 97.5))],
        "n_workers": int(len(workers)),
    }


# ----------------------------------------------------------------------------- PbAS reference


def pbas_arrays() -> dict[str, np.ndarray]:
    """Released per-run, per-trial, per-iteration correctness.

    Shape (2 conditions [unoccluded, occluded], 32 runs, 120 trials, 200 iterations).
    """
    return {
        "PbAS": np.load(BEH / "pbas-full-model-anp.npy"),
        "Pixel-PbAS": np.load(BEH / "td-no-image-encoding-anp.npy"),
    }


def pbas_trial_accuracy(anp: np.ndarray, cond: str, iteration: int) -> np.ndarray:
    c = 0 if cond == "unoccluded" else 1
    return anp[c, :, :, iteration].mean(0)


def pbas_rt_predictor(
    anp: np.ndarray, cond: str, criterion: float, same_mask: np.ndarray | None = None
) -> np.ndarray:
    """Earliest iteration at which the run-averaged accuracy exceeds the criterion (200 if never), per trial.

    The paper sets the criterion to the average participant accuracy of the trial's condition (same vs
    different category); pass ``criterion`` as a scalar or a length-120 array.
    """
    c = 0 if cond == "unoccluded" else 1
    acc = anp[c].mean(0)  # 120 x 200
    crit = np.broadcast_to(np.asarray(criterion, float), (120,))
    out = np.full(120, 200.0)
    for t in range(120):
        hit = np.where(acc[t] >= crit[t])[0]
        if len(hit):
            out[t] = hit[0] + 1
    return out


In [ ]:
%%writefile /content/world-maps/scripts/cloth_extract.py
"""Shape under cloth: embeddings of the 720 stimulus images (and of imagined images, if present).

For every model we store, per layer, one vector per image: the pooled token (CLS or mean) and the patch
tokens average-pooled to 4 x 4 (so a readout can use coarse spatial layout). Output
``results/cloth/features/{model}.npz`` with ``index`` columns (condition, trial, role, source, sample)
and arrays ``X_{layer}`` (float16). ``source`` is ``stim`` for the released images, ``drape`` / ``undrape``
for generator outputs written by ``cloth_imagine.py`` (``results/cloth/imagined/{drape,undrape,drape_video,undrape_video}/...``).

Models: alexnet (fc6, the paper's bottom-up reference; needs torchvision), dinoS/B/L/g (DINOv2),
dinov3L, siglip2, clipL, vjepaL/H/g (video encoder on a 16-frame static clip), and ``vlm`` (Qwen3-VL
2AFC: P(left) for "which object is under the cloth?", stored separately in ``vlm.npz``).
"""

from __future__ import annotations

import argparse
import glob
import json
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import cloth  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

OUT = RESULTS / "cloth" / "features"
IMAGINED = RESULTS / "cloth" / "imagined"
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.4815, 0.4578, 0.4082]).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.2686, 0.2613, 0.2758]).view(1, 3, 1, 1)

MODELS = {
    "alexnet": ("alexnet", None, 224, {}),
    "dinoS": ("hf_image", "facebook/dinov2-small", 224, {"layers": [6, 9, 12]}),
    "dinoB": ("hf_image", "facebook/dinov2-base", 224, {"layers": [6, 9, 12]}),
    "dinoL": ("hf_image", "facebook/dinov2-large", 224, {"layers": [8, 12, 16, 20, 24]}),
    "dinog": ("hf_image", "facebook/dinov2-giant", 224, {"layers": [10, 20, 30, 40]}),
    "dinov3L": ("hf_image", "facebook/dinov3-vitl16-pretrain-lvd1689m", 224, {"layers": [8, 12, 16, 20, 24]}),
    "siglip2": ("hf_image", "google/siglip2-so400m-patch16-256", 256, {"layers": [9, 18, 27], "norm": "half"}),
    "clipL": ("hf_image", "openai/clip-vit-large-patch14", 224, {"layers": [8, 12, 16, 20, 24], "norm": "clip"}),
    "vjepaL": ("vjepa2", "facebook/vjepa2-vitl-fpc64-256", 256, {"layers": [8, 16, 24]}),
    "vjepaH": ("vjepa2", "facebook/vjepa2-vith-fpc64-256", 256, {"layers": [16, 24, 32]}),
    "vjepag": ("vjepa2", "facebook/vjepa2-vitg-fpc64-256", 256, {"layers": [20, 30, 40]}),
    "vlm": ("qwen_vl", "Qwen/Qwen3-VL-8B-Instruct", 224, {}),
}
DEFAULT = ["dinoB", "dinoL", "dinov3L", "siglip2", "clipL", "vjepaL", "alexnet"]


def n_params(m) -> int:
    return int(sum(p.numel() for p in m.parameters()))


def load_rgb(path: str, size: int) -> np.ndarray:
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    if im.shape[0] != size:
        im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    return im


def pool_grid(tok: torch.Tensor, side: int, grid: int = 4) -> torch.Tensor:
    B, N, D = tok.shape
    x = tok.view(B, side, side, D).permute(0, 3, 1, 2)
    if side % grid and x.device.type == "mps":
        x = x.cpu()
    return F.adaptive_avg_pool2d(x, grid).permute(0, 2, 3, 1).reshape(B, -1)


def image_table() -> pd.DataFrame:
    idx = cloth.image_index()
    idx["source"], idx["sample"] = "stim", 0
    rows = [idx]
    for src in ["drape", "undrape", "drape_video", "undrape_video"]:
        for p in sorted(glob.glob(str(IMAGINED / src / "*.png"))):
            name = os.path.basename(p)[:-4]  # trial_{t}_{role}_{k}
            parts = name.split("_")
            rows.append(pd.DataFrame([{"condition": "occluded", "trial": int(parts[1]), "role": parts[2], "path": p,
                                       "same_category": ((int(parts[1]) // 6) % 2 == 0), "source": src, "sample": int(parts[3])}]))
    return pd.concat(rows, ignore_index=True)


# ----------------------------------------------------------------------------- families

def run_hf_image(name, hub, size, opts, table, log):
    from transformers import AutoModel

    kw = {"token": os.environ.get("HF_TOKEN")} if os.environ.get("HF_TOKEN") else {}
    m = AutoModel.from_pretrained(hub, **kw).to(DEV).eval()
    vision = m.vision_model if hasattr(m, "vision_model") else m
    cfg = vision.config if hasattr(vision, "config") else m.config
    patch = getattr(cfg, "patch_size", 14)
    side = size // patch
    per = side * side
    if opts.get("norm") == "half":
        mean, std = torch.tensor([0.5] * 3).view(1, 3, 1, 1), torch.tensor([0.5] * 3).view(1, 3, 1, 1)
    elif opts.get("norm") == "clip":
        mean, std = CLIP_MEAN, CLIP_STD
    else:
        mean, std = MEAN, STD
    layers = opts["layers"]
    feats = {f"L{L}_cls": [] for L in layers}
    feats.update({f"L{L}_grid": [] for L in layers})
    feats["final_pooled"] = []
    bs = 32
    paths = table.path.tolist()
    for s in range(0, len(paths), bs):
        x = np.stack([load_rgb(p, size) for p in paths[s: s + bs]])
        px = ((torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255.0 - mean) / std).to(DEV)
        with torch.no_grad():
            out = vision(pixel_values=px, output_hidden_states=True)
            hs = out.hidden_states
            for L in layers:
                h = hs[min(L, len(hs) - 1)].float()
                patches = h[:, -per:]
                cls = h[:, 0] if h.size(1) > per else patches.mean(1)
                feats[f"L{L}_cls"].append(cls.cpu().half().numpy())
                feats[f"L{L}_grid"].append(pool_grid(patches, side).cpu().half().numpy())
            pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
            feats["final_pooled"].append(pooled.float().cpu().half().numpy())
    log(f"  {name}: {len(paths)} images")
    return {k: np.concatenate(v) for k, v in feats.items()}, n_params(m)


def run_alexnet(name, hub, size, opts, table, log):
    import torchvision

    m = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1).to(DEV).eval()
    fc6 = torch.nn.Sequential(m.features, m.avgpool, torch.nn.Flatten(), *list(m.classifier.children())[:2])  # dropout, fc6
    fc6b = torch.nn.Sequential(fc6, *list(m.classifier.children())[2:5])  # relu, dropout, fc7
    feats = {"fc6": [], "fc7": [], "pool5": []}
    paths = table.path.tolist()
    for s in range(0, len(paths), 32):
        x = np.stack([load_rgb(p, size) for p in paths[s: s + 32]])
        px = ((torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255.0 - MEAN) / STD).to(DEV)
        with torch.no_grad():
            p5 = torch.nn.Flatten()(m.avgpool(m.features(px)))
            f6 = fc6(px)
            f7 = fc6b(px)
        feats["pool5"].append(p5.cpu().half().numpy())
        feats["fc6"].append(f6.cpu().half().numpy())
        feats["fc7"].append(f7.cpu().half().numpy())
    log(f"  {name}: {len(paths)} images")
    return {k: np.concatenate(v) for k, v in feats.items()}, n_params(m)


def run_vjepa(name, hub, size, opts, table, log):
    from transformers import VJEPA2Model

    m = VJEPA2Model.from_pretrained(hub).to(DEV).eval()
    dtype = torch.bfloat16 if DEV == "cuda" else torch.float32
    m = m.to(dtype)
    side = size // 16
    per = side * side
    layers = opts["layers"]
    feats = {f"L{L}_mean": [] for L in layers}
    feats.update({f"L{L}_grid": [] for L in layers})
    paths = table.path.tolist()
    for s in range(0, len(paths), 4):
        x = np.stack([load_rgb(p, size) for p in paths[s: s + 4]])
        px = ((torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255.0 - MEAN) / STD)
        clip = px.unsqueeze(1).repeat(1, 16, 1, 1, 1).to(DEV, dtype)  # static 16-frame clip
        with torch.no_grad():
            out = m(clip, skip_predictor=True, output_hidden_states=True)
            for L in layers:
                h = out.hidden_states[min(L, len(out.hidden_states) - 1)].float()[:, -per:]  # last temporal slot
                feats[f"L{L}_mean"].append(h.mean(1).cpu().half().numpy())
                feats[f"L{L}_grid"].append(pool_grid(h, side).cpu().half().numpy())
    log(f"  {name}: {len(paths)} images")
    return {k: np.concatenate(v) for k, v in feats.items()}, n_params(m)


def run_vlm(name, hub, size, opts, table, log):
    """Qwen3-VL 2AFC on the released trials: P('1') vs P('2') for which candidate is the object under the cloth
    (occluded) or the same object (unoccluded); candidates shown in both orders, averaged."""
    from PIL import Image
    from transformers import AutoModelForImageTextToText, AutoProcessor

    dtype = torch.bfloat16 if DEV != "cpu" else torch.float32
    try:
        m = AutoModelForImageTextToText.from_pretrained(hub, dtype=dtype, device_map={"": DEV}).eval()
    except TypeError:
        m = AutoModelForImageTextToText.from_pretrained(hub, torch_dtype=dtype, device_map={"": DEV}).eval()
    proc = AutoProcessor.from_pretrained(hub)
    tok = proc.tokenizer
    ids1 = {tok.encode(t, add_special_tokens=False)[0] for t in ("1", " 1")}
    ids2 = {tok.encode(t, add_special_tokens=False)[0] for t in ("2", " 2")}
    prompts = {
        "occluded": "The first image shows an object completely covered by a thin draped cloth. The second and third images show "
                    "two different uncovered objects, each in its own random pose. Which uncovered object is the one under the cloth? "
                    "Answer with 1 for the second image or 2 for the third image.",
        "unoccluded": "The first image shows an object. The second and third images show two objects in other random poses. "
                      "Which one is the same object as in the first image? Answer with 1 for the second image or 2 for the third image.",
    }

    def p1(imgs, prompt):
        content = [{"type": "image"} for _ in imgs] + [{"type": "text", "text": prompt}]
        text = proc.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=imgs, return_tensors="pt")
        inputs = {k: (v.to(DEV) if hasattr(v, "to") else v) for k, v in inputs.items()}
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1].float()
        l1, l2 = torch.logsumexp(logits[list(ids1)], 0), torch.logsumexp(logits[list(ids2)], 0)
        return float(torch.softmax(torch.stack([l1, l2]), 0)[0])

    rows = []
    for tr in cloth.trials():
        for cond in cloth.CONDITIONS.values():
            im = tr.images(cond)
            T, M, D = (Image.open(im[k]).convert("RGB") for k in ("target", "match", "distractor"))
            p_match = 0.5 * (p1([T, M, D], prompts[cond]) + (1 - p1([T, D, M], prompts[cond])))
            rows.append({"condition": cond, "trial": tr.t, "p_match": p_match})
        log(f"  {name} trial {tr.t}")
    return pd.DataFrame(rows), n_params(m)


RUNNERS = {"hf_image": run_hf_image, "alexnet": run_alexnet, "vjepa2": run_vjepa, "qwen_vl": run_vlm}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", default=DEFAULT)
    ap.add_argument("--out", default=str(OUT))
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args(argv)
    out = __import__("pathlib").Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    cloth.fetch_assets()
    table = image_table()
    table.to_csv(out / "image_index.csv", index=False)
    meta_path = out / "models.json"
    meta = json.load(open(meta_path)) if meta_path.exists() else {}

    def log(s):
        print(s, flush=True)

    for name in args.models:
        family, hub, size, opts = MODELS[name]
        f = out / f"{name}.npz"
        if f.exists() and not args.force and meta.get(name, {}).get("n_images") == len(table):
            log(f"{name}: done, skipping")
            continue
        t0 = time.time()
        try:
            res, npar = RUNNERS[family](name, hub, size, opts, table, log)
            if isinstance(res, pd.DataFrame):
                res.to_csv(out / f"{name}.csv", index=False)
                dims = {"p_match": 1}
            else:
                np.savez(f, **{f"X_{k}": v for k, v in res.items()})
                dims = {k: int(v.shape[1]) for k, v in res.items()}
            meta[name] = {"status": "ok", "hub": hub, "params": npar, "device": DEV, "n_images": len(table), "dims": dims,
                          "seconds": round(time.time() - t0, 1)}
            if DEV == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            import traceback
            traceback.print_exc()
            meta[name] = {"status": f"failed: {type(e).__name__}: {str(e)[:300]}", "hub": hub}
        with open(meta_path, "w") as fh:
            json.dump(meta, fh, indent=1)
        log(f"{name}: {meta[name]['status']} ({meta[name].get('seconds', 0)}s)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/world-maps/scripts/cloth_analyze.py
"""Shape under cloth: humans, PbAS and modern models on the same trials.

Readouts
  BU (paper protocol): d1 = corr(e_target, e_match), d2 = corr(e_target, e_distractor); choice = d1 > d2;
     continuous score = Luce rule on (1 + corr) / 2; margin = |d1 - d2| as a confidence proxy for RT.
  De-draping (new, D-style): ridge map from the draped target's embedding to the bare object's embedding,
     learned on the other trials (leave-one-trial-out) or on the other categories (leave-category-out),
     then the same 2AFC on the mapped embedding. Tests whether the hidden shape is linearly recoverable
     from the features even when raw similarity does not use it.
  Imagination (new): generator-in-the-loop analysis-by-synthesis. 'drape': imagined draped versions of each
     candidate compared with the target; 'undrape': imagined bare objects from the target compared with the
     candidates. Sequential sampling over imagined samples gives an RT prediction (samples to criterion),
     as PbAS's iterations do.
  VLM: Qwen3-VL 2AFC probability.

Human side follows the lab's scripts (exclusions, per-trial accuracy, RT z-filter, participant bootstrap,
split-half ceiling). PbAS and Pixel-PbAS per-trial predictions come from the released run arrays.
Outputs: results/cloth/cloth_results.json / .md, figures/*.png
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import RidgeCV

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import cloth  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

warnings.filterwarnings("ignore", category=RuntimeWarning)
ALPHAS = np.logspace(-2, 6, 17)
SAME = np.array([((t // 6) % 2 == 0) for t in range(120)])


def corr_rows(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Row-wise Pearson correlation between matched rows of a and b."""
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)


def luce(d1, d2):
    p1, p2 = (1 + d1) / 2, (1 + d2) / 2
    return p1 / (p1 + p2 + 1e-9)


def boot_corr(model: np.ndarray, human_boot: np.ndarray, sel=None) -> dict:
    """Correlation between a per-trial model score and participant-bootstrapped human accuracy."""
    sel = np.arange(120) if sel is None else np.asarray(sel)
    m = model[sel]
    if m.std() == 0:
        return {"mean": float("nan"), "ci": [float("nan"), float("nan")], "point": float("nan")}
    rs = np.array([np.corrcoef(m, hb[sel])[0, 1] if hb[sel].std() > 0 else np.nan for hb in human_boot])
    point = float(np.corrcoef(m, human_boot.mean(0)[sel])[0, 1])
    return {"mean": float(np.nanmean(rs)), "ci": [float(np.nanpercentile(rs, 2.5)), float(np.nanpercentile(rs, 97.5))], "point": point, "_rs": rs}


def compare_boot(rs_a: np.ndarray, rs_b: np.ndarray) -> dict:
    d = rs_a - rs_b
    d = d[~np.isnan(d)]
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return {"diff_mean": float(d.mean()), "diff_ci": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))], "p_two_sided": float(min(1.0, p))}


# ----------------------------------------------------------------------------- features

class Feats:
    def __init__(self, path: Path, index: pd.DataFrame):
        z = np.load(path)
        self.layers = [k[2:] for k in z.files if k.startswith("X_")]
        self.X = {k[2:]: z[k].astype(np.float32) for k in z.files if k.startswith("X_")}
        self.index = index.reset_index(drop=True)
        self.lookup = {}
        for i, r in self.index.iterrows():
            self.lookup[(r.condition, int(r.trial), r.role, r.source, int(r["sample"]))] = i

    def get(self, layer, condition, trial, role, source="stim", sample=0):
        i = self.lookup.get((condition, trial, role, source, sample))
        return None if i is None else self.X[layer][i]

    def triplets(self, layer, condition):
        T = np.array([self.get(layer, condition, t, "target") for t in range(120)])
        M = np.array([self.get(layer, condition, t, "match") for t in range(120)])
        D = np.array([self.get(layer, condition, t, "distractor") for t in range(120)])
        return T, M, D

    def samples(self, layer, source, role):
        """Imagined samples: dict trial -> array (K, D)."""
        out = {}
        for t in range(120):
            ks = [k for (c, tt, r, s, k) in self.lookup if tt == t and r == role and s == source]
            if ks:
                out[t] = np.array([self.get(layer, "occluded", t, role, source, k) for k in sorted(ks)])
        return out


# ----------------------------------------------------------------------------- readouts

def bu_scores(T, M, D):
    d1, d2 = corr_rows(T, M), corr_rows(T, D)
    return {"d1": d1, "d2": d2, "correct": (d1 > d2).astype(float), "score": luce(d1, d2), "margin": np.abs(d1 - d2)}


def cell_accuracy(correct: np.ndarray) -> dict:
    return {"all": float(correct.mean()), "same_category": float(correct[SAME].mean()), "different_category": float(correct[~SAME].mean())}


def dedrape(T, M, D, mode="loo", k_in=100, k_out=64):
    """Ridge map draped-target -> bare-match embedding, evaluated on held-out trials (loo) or categories (loco).

    Inputs and outputs are compressed with PCA fitted without labels (targets: the 120 draped images; outputs:
    the 240 bare candidates), so the readout is a 100 -> 64 linear map and the 2AFC is scored in the
    64-d bare-object space.
    """
    from sklearn.decomposition import PCA

    p_in = PCA(min(k_in, T.shape[0] - 1, T.shape[1])).fit(T)
    p_out = PCA(min(k_out, 2 * M.shape[0] - 1, M.shape[1])).fit(np.vstack([M, D]))
    Tz, Mz, Dz = p_in.transform(T), p_out.transform(M), p_out.transform(D)
    pred = np.zeros_like(Mz)
    groups = np.arange(120) if mode == "loo" else np.array([t // 12 for t in range(120)])  # 10 categories x 12 trials
    for g in np.unique(groups):
        te = groups == g
        m = RidgeCV(alphas=ALPHAS).fit(Tz[~te], Mz[~te])
        pred[te] = m.predict(Tz[te])
    return bu_scores(pred, Mz, Dz)


def sequential_rt(ev: np.ndarray, target_acc: float, repeats: int = 50, seed: int = 0) -> dict:
    """ev: (120, K) per-sample evidence (match minus distractor). Accumulate in random order until |sum| >= theta.

    theta is chosen so that the mean decision accuracy over trials matches target_acc (grid search).
    Returns per-trial mean number of samples to decision and accuracy.
    """
    rng = np.random.default_rng(seed)
    K = ev.shape[1]
    best = None
    for theta in np.linspace(0.0, np.abs(ev).sum(1).max(), 60):
        acc, nsamp = np.zeros(120), np.zeros(120)
        for r in range(repeats):
            for t in range(120):
                order = rng.permutation(K)
                s = 0.0
                n = K
                for j, k in enumerate(order):
                    s += ev[t, k]
                    if abs(s) >= theta:
                        n = j + 1
                        break
                acc[t] += float(s > 0) if s != 0 else 0.5
                nsamp[t] += n
        acc, nsamp = acc / repeats, nsamp / repeats
        if best is None or abs(acc.mean() - target_acc) < abs(best[0] - target_acc):
            best = (acc.mean(), theta, acc.copy(), nsamp.copy())
    return {"theta": float(best[1]), "mean_acc": float(best[0]), "acc": best[2], "n_samples": best[3]}


# ----------------------------------------------------------------------------- main

def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--features", default=str(RESULTS / "cloth" / "features"))
    ap.add_argument("--out", default=str(RESULTS / "cloth"))
    ap.add_argument("--models", nargs="+", default=None)
    ap.add_argument("--n-boot", type=int, default=2000)
    args = ap.parse_args(argv)
    feat, out = Path(args.features), Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    cloth.fetch_assets()

    # ---- humans
    df = cloth.load_behaviour()
    ht = cloth.human_trial_table(df)
    human = {"n_workers_kept": int(df.workerid.nunique()), "cells": {}, "ceiling": {}, "trial_table": ht.to_dict("records")}
    acc = {}
    boot = {}
    for cond in ["unoccluded", "occluded"]:
        for time in ["1s", "2s", "unlimited"]:
            sub = ht[(ht.cond == cond) & (ht.time == time)].set_index("trial").reindex(range(120))
            a = sub.acc.values
            acc[(cond, time)] = a
            human["cells"][f"{cond}/{time}"] = {"all": float(np.nanmean(a)), "same_category": float(np.nanmean(a[SAME])),
                                                "different_category": float(np.nanmean(a[~SAME])), "n_workers": int(df[(df.cond == cond) & (df.time == time)].workerid.nunique()),
                                                "mean_rt_ms": float(np.nanmean(sub.rt.values))}
            boot[(cond, time)] = cloth.participant_bootstrap(df, cond, time, n_boot=args.n_boot)
        human["ceiling"][cond] = {"all": cloth.split_half_ceiling(df, cond, "unlimited"),
                                  "same_category": cloth.split_half_ceiling(df, cond, "unlimited", which=np.where(SAME)[0])}
    log_rt = {cond: ht[(ht.cond == cond) & (ht.time == "unlimited")].set_index("trial").reindex(range(120)).log_rt.values for cond in ["unoccluded", "occluded"]}

    # ---- PbAS reference from the released runs
    ref = {}
    anps = cloth.pbas_arrays()
    for name, anp in anps.items():
        iters = cloth.PBAS_ITER if name == "PbAS" else cloth.PIXEL_PBAS_ITER
        ref[name] = {}
        for cond in ["unoccluded", "occluded"]:
            e = {}
            for time, it in iters.items():
                pa = cloth.pbas_trial_accuracy(anp, cond, it)
                e[time] = {"cells": cell_accuracy(pa), "r_all": {k: v for k, v in boot_corr(pa, boot[(cond, time)]).items() if k != "_rs"},
                           "r_same": {k: v for k, v in boot_corr(pa, boot[(cond, time)], np.where(SAME)[0]).items() if k != "_rs"}}
                if time == "unlimited":
                    e["_rs_all"], e["_rs_same"] = boot_corr(pa, boot[(cond, time)])["_rs"], boot_corr(pa, boot[(cond, time)], np.where(SAME)[0])["_rs"]
            crit = float(np.nanmean(acc[(cond, "unlimited")]))  # the lab's rule: one criterion per occlusion condition
            rt_pred = cloth.pbas_rt_predictor(anp, cond, crit)
            ok = ~np.isnan(log_rt[cond])
            e["rt_r_log"] = float(np.corrcoef(np.log(rt_pred[ok]), log_rt[cond][ok])[0, 1])
            ref[name][cond] = e

    # ---- models
    index = pd.read_csv(feat / "image_index.csv")
    meta = json.load(open(feat / "models.json")) if (feat / "models.json").exists() else {}
    names = [n for n, v in meta.items() if v.get("status") == "ok" and n != "vlm" and (feat / f"{n}.npz").exists()]
    if args.models:
        names = [n for n in names if n in args.models]
    models = {}
    for name in names:
        F = Feats(feat / f"{name}.npz", index)
        models[name] = {"params": meta[name].get("params"), "layers": {}}
        for layer in F.layers:
            entry = {}
            for cond in ["unoccluded", "occluded"]:
                T, M, D = F.triplets(layer, cond)
                bu = bu_scores(T, M, D)
                hb = boot[(cond, "unlimited")]
                r_all, r_same = boot_corr(bu["score"], hb), boot_corr(bu["score"], hb, np.where(SAME)[0])
                ok = ~np.isnan(log_rt[cond])
                e = {"cells": cell_accuracy(bu["correct"]),
                     "r_all": {k: v for k, v in r_all.items() if k != "_rs"}, "r_same": {k: v for k, v in r_same.items() if k != "_rs"},
                     "vs_pbas_same": compare_boot(r_same["_rs"], ref["PbAS"][cond]["_rs_same"]),
                     "margin_vs_log_rt": {"pearson": float(np.corrcoef(bu["margin"][ok], log_rt[cond][ok])[0, 1]),
                                          "spearman": float(stats.spearmanr(bu["margin"][ok], log_rt[cond][ok]).correlation)},
                     "r_all_1s": boot_corr(bu["score"], boot[(cond, "1s")])["mean"], "r_all_2s": boot_corr(bu["score"], boot[(cond, "2s")])["mean"]}
                if cond == "occluded":
                    for mode in ["loo", "loco"]:
                        dd = dedrape(T, M, D, mode)
                        e[f"dedrape_{mode}"] = {"cells": cell_accuracy(dd["correct"]),
                                                "r_all": {k: v for k, v in boot_corr(dd["score"], hb).items() if k != "_rs"},
                                                "r_same": {k: v for k, v in boot_corr(dd["score"], hb, np.where(SAME)[0]).items() if k != "_rs"}}
                    # imagination readouts
                    for src in ["drape", "undrape", "drape_video", "undrape_video"]:
                        if src.startswith("drape"):
                            SM, SD = F.samples(layer, src, "match"), F.samples(layer, src, "distractor")
                            if len(SM) < 120 or len(SD) < 120:
                                continue
                            K = min(min(v.shape[0] for v in SM.values()), min(v.shape[0] for v in SD.values()))
                            ev = np.array([[corr_rows(T[t:t + 1], SM[t][k:k + 1])[0] - corr_rows(T[t:t + 1], SD[t][k:k + 1])[0] for k in range(K)] for t in range(120)])
                        else:
                            SU = F.samples(layer, src, "target")
                            if len(SU) < 120:
                                continue
                            K = min(v.shape[0] for v in SU.values())
                            ev = np.array([[corr_rows(SU[t][k:k + 1], M[t:t + 1])[0] - corr_rows(SU[t][k:k + 1], D[t:t + 1])[0] for k in range(K)] for t in range(120)])
                        mean_ev = ev.mean(1)
                        correct = (mean_ev > 0).astype(float)
                        score = 1 / (1 + np.exp(-mean_ev / (np.abs(mean_ev).std() + 1e-9)))
                        comb = 0.5 * (stats.zscore(mean_ev) + stats.zscore(bu["d1"] - bu["d2"]))
                        rs = boot_corr(score, hb, np.where(SAME)[0])
                        rt = sequential_rt(ev, np.nanmean(acc[("occluded", "unlimited")]))
                        e[f"imagine_{src}"] = {"K": int(K), "cells": cell_accuracy(correct),
                                               "r_all": {k: v for k, v in boot_corr(score, hb).items() if k != "_rs"},
                                               "r_same": {k: v for k, v in rs.items() if k != "_rs"},
                                               "vs_bu_same": compare_boot(rs["_rs"], r_same["_rs"]),
                                               "vs_pbas_same": compare_boot(rs["_rs"], ref["PbAS"]["occluded"]["_rs_same"]),
                                               "combined_with_bu": {"cells": cell_accuracy((comb > 0).astype(float)),
                                                                    "r_same": {k: v for k, v in boot_corr(comb, hb, np.where(SAME)[0]).items() if k != "_rs"}},
                                               "sequential_rt": {"theta": rt["theta"], "mean_acc": rt["mean_acc"],
                                                                 "r_log_rt": float(np.corrcoef(np.log(rt["n_samples"][ok]), log_rt["occluded"][ok])[0, 1]),
                                                                 "spearman_rt": float(stats.spearmanr(rt["n_samples"][ok], log_rt["occluded"][ok]).correlation)}}
                entry[cond] = e
            models[name]["layers"][layer] = entry
        print(f"{name}: {len(F.layers)} layers", flush=True)

    # ---- VLM
    vlm = {}
    if (feat / "vlm.csv").exists():
        v = pd.read_csv(feat / "vlm.csv")
        for cond in ["unoccluded", "occluded"]:
            p = v[v.condition == cond].set_index("trial").reindex(range(120)).p_match.values
            hb = boot[(cond, "unlimited")]
            vlm[cond] = {"cells": cell_accuracy((p > 0.5).astype(float)), "r_all": {k: vv for k, vv in boot_corr(p, hb).items() if k != "_rs"},
                         "r_same": {k: vv for k, vv in boot_corr(p, hb, np.where(SAME)[0]).items() if k != "_rs"}}

    # ---- best layer per model (by occluded same-category r) and the scale table
    summary_rows = []
    for name, m in models.items():
        best = max(m["layers"].items(), key=lambda kv: kv[1]["occluded"]["r_same"]["mean"] if np.isfinite(kv[1]["occluded"]["r_same"]["mean"]) else -9)
        L, e = best
        summary_rows.append({"model": name, "params": m["params"], "best_layer": L,
                             "occ_acc_same": e["occluded"]["cells"]["same_category"], "occ_acc_diff": e["occluded"]["cells"]["different_category"],
                             "unocc_acc_same": e["unoccluded"]["cells"]["same_category"],
                             "occ_r_same": e["occluded"]["r_same"]["mean"], "occ_r_same_ci": e["occluded"]["r_same"]["ci"],
                             "occ_r_all": e["occluded"]["r_all"]["mean"], "unocc_r_same": e["unoccluded"]["r_same"]["mean"],
                             "occ_margin_rt_spearman": e["occluded"]["margin_vs_log_rt"]["spearman"],
                             "dedrape_loo_acc_same": e["occluded"].get("dedrape_loo", {}).get("cells", {}).get("same_category"),
                             "dedrape_loco_acc_same": e["occluded"].get("dedrape_loco", {}).get("cells", {}).get("same_category"),
                             "dedrape_loco_r_same": e["occluded"].get("dedrape_loco", {}).get("r_same", {}).get("mean"),
                             "imagine_drape_acc_same": e["occluded"].get("imagine_drape", {}).get("cells", {}).get("same_category"),
                             "imagine_drape_r_same": e["occluded"].get("imagine_drape", {}).get("r_same", {}).get("mean"),
                             "imagine_undrape_r_same": e["occluded"].get("imagine_undrape", {}).get("r_same", {}).get("mean"),
                             "imagine_drape_video_acc_same": e["occluded"].get("imagine_drape_video", {}).get("cells", {}).get("same_category"),
                             "imagine_drape_video_r_same": e["occluded"].get("imagine_drape_video", {}).get("r_same", {}).get("mean"),
                             "imagine_undrape_video_r_same": e["occluded"].get("imagine_undrape_video", {}).get("r_same", {}).get("mean"),
                             "vs_pbas_p": e["occluded"]["vs_pbas_same"]["p_two_sided"]})
    summary = pd.DataFrame(summary_rows)

    res = {"human": {k: v for k, v in human.items() if k != "trial_table"}, "reference": {n: {c: {k: v for k, v in e.items() if not k.startswith("_")} for c, e in d.items()} for n, d in ref.items()},
           "models": models, "vlm": vlm, "summary": summary.to_dict("records")}
    figures(res, human, ref, models, out / "figures")
    with open(out / "cloth_results.json", "w") as f:
        json.dump(res, f, indent=1, default=lambda o: None if isinstance(o, float) and np.isnan(o) else (o.item() if hasattr(o, "item") else str(o)))
    md = summarize(res, summary)
    (out / "cloth_results.md").write_text(md)
    print(md)


def summarize(res, summary: pd.DataFrame) -> str:
    h = res["human"]
    f = lambda v, d=2: "–" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:.{d}f}"
    lines = ["# Shape under cloth: humans, PbAS and modern models", ""]
    lines.append(f"Humans: {h['n_workers_kept']} workers kept. Unlimited-time accuracy, occluded: all {f(h['cells']['occluded/unlimited']['all'])}, "
                 f"same-category {f(h['cells']['occluded/unlimited']['same_category'])}, different-category {f(h['cells']['occluded/unlimited']['different_category'])}; "
                 f"unoccluded same-category {f(h['cells']['unoccluded/unlimited']['same_category'])}.")
    lines.append(f"Split-half ceiling (unlimited): occluded all {f(h['ceiling']['occluded']['all']['mean'])}, occluded same-category "
                 f"{f(h['ceiling']['occluded']['same_category']['mean'])}, unoccluded same-category {f(h['ceiling']['unoccluded']['same_category']['mean'])}.")
    r = res["reference"]
    lines.append(f"PbAS (released runs, iteration 110): occluded same-category acc {f(r['PbAS']['occluded']['unlimited']['cells']['same_category'])}, "
                 f"trial r same-category {f(r['PbAS']['occluded']['unlimited']['r_same']['mean'])} [{f(r['PbAS']['occluded']['unlimited']['r_same']['ci'][0])}, {f(r['PbAS']['occluded']['unlimited']['r_same']['ci'][1])}], "
                 f"all trials {f(r['PbAS']['occluded']['unlimited']['r_all']['mean'])}; unoccluded same-category r {f(r['PbAS']['unoccluded']['unlimited']['r_same']['mean'])}; "
                 f"RT r (log) occluded {f(r['PbAS']['occluded']['rt_r_log'])}, unoccluded {f(r['PbAS']['unoccluded']['rt_r_log'])}. Paper: 0.44 / 0.44, RT 0.71 / 0.67.")
    lines += ["", "| model | params | best layer | occluded acc same / diff | unoccluded acc same | occluded r same [CI] | occluded r all | unocc r same | margin–RT ρ | de-drape acc same (loo / loco) | de-drape loco r same | imagine drape acc / r same | imagine undrape r same | video drape acc / r same | video undrape r same | p vs PbAS |",
              "|" + "---|" * 16]
    for _, s in summary.sort_values("occ_r_same", ascending=False).iterrows():
        lines.append(f"| {s.model} | {f(s.params/1e6,0) if s.params else '–'}M | {s.best_layer} | {f(s.occ_acc_same)} / {f(s.occ_acc_diff)} | {f(s.unocc_acc_same)} | "
                     f"{f(s.occ_r_same)} [{f(s.occ_r_same_ci[0])}, {f(s.occ_r_same_ci[1])}] | {f(s.occ_r_all)} | {f(s.unocc_r_same)} | {f(s.occ_margin_rt_spearman)} | "
                     f"{f(s.dedrape_loo_acc_same)} / {f(s.dedrape_loco_acc_same)} | {f(s.dedrape_loco_r_same)} | {f(s.imagine_drape_acc_same)} / {f(s.imagine_drape_r_same)} | {f(s.imagine_undrape_r_same)} | "
                     f"{f(s.imagine_drape_video_acc_same)} / {f(s.imagine_drape_video_r_same)} | {f(s.imagine_undrape_video_r_same)} | {f(s.vs_pbas_p,3)} |")
    if res["vlm"]:
        lines += ["", "VLM (Qwen3-VL 2AFC): " + json.dumps({c: {"acc_same": v["cells"]["same_category"], "acc_diff": v["cells"]["different_category"], "r_same": v["r_same"]["mean"]} for c, v in res["vlm"].items()})]
    lines += ["", "Readouts: BU = paper protocol on frozen embeddings; de-drape = linear map draped→bare learned on other trials (loo) or other categories (loco); "
              "imagine = generator-in-the-loop analysis-by-synthesis (if imagined images were produced). r values are means over participant bootstraps; CI 95%."]
    return "\n".join(lines)


def figures(res, human, ref, models, out: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    out.mkdir(parents=True, exist_ok=True)
    summ = pd.DataFrame(res["summary"])
    if len(summ) == 0:
        return
    summ = summ.sort_values("occ_r_same", ascending=False)
    # trial-level r on occluded same-category trials
    fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(summ) + 3), 4))
    x = np.arange(len(summ))
    ax.bar(x, summ.occ_r_same, yerr=[summ.occ_r_same - summ.occ_r_same_ci.apply(lambda c: c[0]), summ.occ_r_same_ci.apply(lambda c: c[1]) - summ.occ_r_same], color="#36c", label="BU readout")
    if summ.dedrape_loco_r_same.notna().any():
        ax.scatter(x, summ.dedrape_loco_r_same, color="#e80", zorder=3, label="de-drape (held-out category)")
    if summ.imagine_drape_r_same.notna().any():
        ax.scatter(x, summ.imagine_drape_r_same, color="#2a2", marker="s", zorder=3, label="imagination (image: drape)")
    if summ.imagine_drape_video_r_same.notna().any():
        ax.scatter(x, summ.imagine_drape_video_r_same, color="#a2a", marker="^", zorder=3, label="imagination (video: cloth falls)")
    ax.axhline(ref["PbAS"]["occluded"]["unlimited"]["r_same"]["mean"], ls="--", c="k", label="PbAS")
    ax.axhline(human["ceiling"]["occluded"]["same_category"]["mean"], ls=":", c="gray", label="split-half ceiling")
    ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{l}" for m, l in zip(summ.model, summ.best_layer)], fontsize=7)
    ax.set_ylabel("r with human accuracy (occluded, same-category)"); ax.legend(fontsize=7); ax.set_ylim(-0.3, 0.9)
    fig.tight_layout(); fig.savefig(out / "trial_correlation_occluded_same.png", dpi=150); plt.close(fig)
    # cell accuracies
    fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(summ) + 3), 4))
    w = 0.25
    ax.bar(x - w, summ.occ_acc_same, w, label="occluded same-category", color="#c33")
    ax.bar(x, summ.occ_acc_diff, w, label="occluded different-category", color="#e99")
    ax.bar(x + w, summ.unocc_acc_same, w, label="unoccluded same-category", color="#9bd")
    hc = human["cells"]
    ax.axhline(hc["occluded/unlimited"]["same_category"], ls="--", c="#c33", lw=1, label="humans occluded same (unlimited)")
    ax.axhline(0.5, c="gray", lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels(summ.model, fontsize=8); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); ax.set_ylim(0.3, 1.02)
    fig.tight_layout(); fig.savefig(out / "cell_accuracy.png", dpi=150); plt.close(fig)
    # layer sweeps
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, m in models.items():
        ls = list(m["layers"].items())
        ax.plot(range(len(ls)), [e["occluded"]["r_same"]["mean"] for _, e in ls], marker="o", label=name)
    ax.axhline(ref["PbAS"]["occluded"]["unlimited"]["r_same"]["mean"], ls="--", c="k")
    ax.set_xlabel("layer index (as stored)"); ax.set_ylabel("r (occluded, same-category)"); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(out / "layer_sweep.png", dpi=150); plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/world-maps/scripts/cloth_imagine.py
"""Imagination as simulation: a learned image generator in the loop of analysis-by-synthesis (GPU).

Two directions, both on the occluded condition's 120 trials:
  drape    for each candidate (match, distractor; bare object in its own pose) imagine K cloth-draped
           versions  ->  results/cloth/imagined/drape/trial_{t}_{match|distractor}_{k}.png
  undrape  for each draped target imagine K bare objects underneath
           ->  results/cloth/imagined/undrape/trial_{t}_target_{k}.png
``cloth_extract.py`` then embeds these images (source = drape / undrape) and ``cloth_analyze.py`` scores them
against the human data (imagination readouts, sequential-sampling RT).

Generators (diffusers):
  sdxl     stabilityai/stable-diffusion-xl-base-1.0 img2img (not gated; keeps silhouette and pose at moderate
           strength; about 2-4 s per image on an A100 at 768 px)
  kontext  black-forest-labs/FLUX.1-Kontext-dev instruction editing (gated: accept the licence on the Hub
           and set HF_TOKEN; about 10-20 s per image; better instruction following)
  ltx      LTX-Video 2B (Lightricks/LTX-Video-0.9.5; the 0.9.7/0.9.8 "dev" checkpoints are 13B, do not use them on a
           40 GB card) image-to-video: the cloth FALLS over the candidate (or is PULLED OFF the target) and the last
           frame is the imagined image; about 10-20 s per clip at 512 px. Directions are written as drape_video /
           undrape_video so the analysis can compare process (video) with appearance (image).
  wan      Wan-AI/Wan2.2-TI2V-5B-Diffusers image-to-video, same idea, slower (about 1-2 min per clip).
The prompts never name the object category, so no category information leaks into the imagined images.
Resumable: existing files are skipped.
"""

from __future__ import annotations

import argparse
import os
import sys
import time

import torch
from PIL import Image

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src"))
from world_maps import cloth  # noqa: E402
from world_maps.paths import RESULTS  # noqa: E402

OUT = RESULTS / "cloth" / "imagined"
DEV = "cuda" if torch.cuda.is_available() else "cpu"

PROMPTS = {
    "drape": "the same object completely covered by a thin white cotton cloth draped over it, cloth falling naturally with folds and "
             "wrinkles, same camera angle, same pose, same size, matte grey material, uniform black background, 3D render",
    "undrape": "the rigid object that was hidden under the cloth, now uncovered, bare matte grey 3D render, same camera angle, same pose, "
               "same size, uniform black background",
}
NEGATIVE = "text, watermark, colour, people, hands, extra objects, blurry"
VIDEO = {
    "drape": "A thin white cotton cloth falls from above onto the grey object and drapes over it completely, settling with natural "
             "folds and wrinkles until the object is fully covered. Fixed camera, uniform black background, matte grey render, no "
             "other objects.",
    "undrape": "The white cloth covering the object is lifted away and slides off, revealing the bare rigid grey object underneath "
               "in the same position. Fixed camera, uniform black background, matte grey render, no other objects.",
}
VIDEO_DIRS = {"drape": "drape_video", "undrape": "undrape_video"}
KONTEXT = {
    "drape": "Cover this object completely with a thin white cotton cloth draped over it so that it is fully hidden; keep the camera "
             "angle, pose, size and the black background exactly the same.",
    "undrape": "Remove the cloth and show the rigid grey object that was underneath it; keep the camera angle, pose, size and the "
               "black background exactly the same.",
}


def load_pipe(kind: str):
    if kind == "sdxl":
        from diffusers import StableDiffusionXLImg2ImgPipeline

        pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16,
                                                                variant="fp16", use_safetensors=True).to(DEV)
        if hasattr(pipe.vae, "enable_slicing"):
            pipe.vae.enable_slicing()
        return pipe
    if kind == "kontext":
        from diffusers import FluxKontextPipeline

        pipe = FluxKontextPipeline.from_pretrained("black-forest-labs/FLUX.1-Kontext-dev", torch_dtype=torch.bfloat16,
                                                   token=os.environ.get("HF_TOKEN"))
        pipe.enable_model_cpu_offload()
        return pipe
    if kind == "ltx":
        from diffusers import LTXImageToVideoPipeline

        pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video-0.9.5", torch_dtype=torch.bfloat16).to(DEV)
        for fn in ("enable_tiling", "enable_slicing"):
            if hasattr(pipe.vae, fn):
                getattr(pipe.vae, fn)()
        print(f"LTX transformer: {sum(p.numel() for p in pipe.transformer.parameters()) / 1e9:.1f}B parameters", flush=True)
        return pipe
    if kind == "wan":
        from diffusers import AutoencoderKLWan, WanImageToVideoPipeline

        rep = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
        vae = AutoencoderKLWan.from_pretrained(rep, subfolder="vae", torch_dtype=torch.float32)
        pipe = WanImageToVideoPipeline.from_pretrained(rep, vae=vae, torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()
        return pipe
    raise ValueError(kind)


def generate(pipe, kind: str, direction: str, image: Image.Image, seed: int, size: int, strength: float, steps: int,
             n_frames: int = 49, keep_video: str | None = None) -> Image.Image:
    g = torch.Generator(device="cpu").manual_seed(seed)
    src = image.convert("RGB").resize((size, size), Image.BICUBIC)
    if kind == "sdxl":
        return pipe(prompt=PROMPTS[direction], negative_prompt=NEGATIVE, image=src, strength=strength, guidance_scale=6.0,
                    num_inference_steps=steps, generator=g).images[0]
    if kind == "kontext":
        return pipe(image=src, prompt=KONTEXT[direction], guidance_scale=2.5, num_inference_steps=steps, height=size, width=size,
                    generator=g).images[0]
    # image-to-video: the last frame is the imagined end state of the physical process
    kw = dict(image=src, prompt=VIDEO[direction], negative_prompt=NEGATIVE + ", camera motion, zoom", width=size, height=size,
              num_frames=n_frames, num_inference_steps=steps, generator=g)
    if kind == "wan":
        kw["guidance_scale"] = 5.0
    frames = pipe(**kw).frames[0]
    if keep_video:
        try:
            from diffusers.utils import export_to_video

            export_to_video([f if isinstance(f, Image.Image) else Image.fromarray(f) for f in frames], keep_video, fps=24)
        except Exception as e:  # inspection only
            print(f"  (video export skipped: {type(e).__name__})", flush=True)
    last = frames[-1]
    return last if isinstance(last, Image.Image) else Image.fromarray(last)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--generator", default="sdxl", choices=["sdxl", "kontext", "ltx", "wan"])
    ap.add_argument("--n-frames", type=int, default=49, help="frames per clip for the video generators (8k+1 for LTX, 4k+1 for Wan)")
    ap.add_argument("--keep-videos", type=int, default=6, help="save the first N clips per direction as mp4 for inspection")
    ap.add_argument("--directions", nargs="+", default=["drape", "undrape"])
    ap.add_argument("--k", type=int, default=4, help="imagined samples per image")
    ap.add_argument("--size", type=int, default=768)
    ap.add_argument("--steps", type=int, default=None, help="denoising steps (default 30 for image models, 25 for video)")
    ap.add_argument("--strength-drape", type=float, default=0.6)
    ap.add_argument("--strength-undrape", type=float, default=0.7)
    ap.add_argument("--limit", type=int, default=None, help="first N trials (smoke test)")
    ap.add_argument("--out", default=str(OUT))
    args = ap.parse_args(argv)
    out = __import__("pathlib").Path(args.out)
    if args.steps is None:
        args.steps = 25 if args.generator in ("ltx", "wan") else 30
    cloth.fetch_assets()
    pipe = load_pipe(args.generator)
    trials = cloth.trials()[: args.limit] if args.limit else cloth.trials()
    t0 = time.time()
    n = 0
    is_video = args.generator in ("ltx", "wan")
    if is_video and args.size % 32:
        args.size = 32 * (args.size // 32)
        print(f"size rounded to {args.size} for the video generator")
    for direction in args.directions:
        d = out / (VIDEO_DIRS[direction] if is_video else direction)
        d.mkdir(parents=True, exist_ok=True)
        roles = ["match", "distractor"] if direction == "drape" else ["target"]
        strength = args.strength_drape if direction == "drape" else args.strength_undrape
        for tr in trials:
            ims = tr.images("occluded")
            for role in roles:
                for k in range(args.k):
                    f = d / f"trial_{tr.t}_{role}_{k}.png"
                    if f.exists():
                        continue
                    try:
                        keep = str(f.with_suffix(".mp4")) if (is_video and n < args.keep_videos) else None
                        img = generate(pipe, args.generator, direction, Image.open(ims[role]), seed=1000 * tr.t + 10 * k + (role == "distractor"),
                                       size=args.size, strength=strength, steps=args.steps, n_frames=args.n_frames, keep_video=keep)
                        img.resize((224, 224), Image.LANCZOS).save(f)
                        n += 1
                    except Exception as e:  # keep going
                        print(f"  failed {f.name}: {type(e).__name__}: {str(e)[:120]}", flush=True)
            if tr.t % 10 == 0:
                print(f"{direction} trial {tr.t}: {n} images so far, {time.time() - t0:.0f}s", flush=True)
    print(f"done: {n} new images in {time.time() - t0:.0f}s -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
#@title 3. Smoke test: one clip per direction, shown as video + last frame
%cd /content/world-maps
if SMOKE_TEST_FIRST:
    !PYTHONPATH=src python scripts/cloth_imagine.py --generator {VIDEO_GENERATOR} --k 1 --limit 1 --size {VIDEO_SIZE} --n-frames {N_FRAMES} --keep-videos 4 --out /content/smoke_video 2>&1 | tail -4
    from IPython.display import display, Video
    from PIL import Image
    import glob
    for f in sorted(glob.glob("/content/smoke_video/*/*.mp4")):
        print(f); display(Video(f, embed=True, width=256))
    for f in sorted(glob.glob("/content/smoke_video/*/*.png")):
        print(f); display(Image.open(f).resize((160, 160)))
else:
    print("skipped")


In [ ]:
#@title 4. Video imagination for all trials (long)
%cd /content/world-maps
limit = f"--limit {IMAGINE_LIMIT}" if IMAGINE_LIMIT else ""
dirs = " ".join(DIRECTIONS)
!PYTHONPATH=src python scripts/cloth_imagine.py --generator {VIDEO_GENERATOR} --directions {dirs} --k {K_SAMPLES} --size {VIDEO_SIZE} --n-frames {N_FRAMES} {limit} 2>&1 | tail -20
import glob, os, numpy as np
from PIL import Image
from IPython.display import display
rows = []
for t in [3, 9, 15, 27]:
    d = "data/cloth/stimuli/unoccluded-occluded"
    ims = [f"{d}/trial_{t}_gtCloth.png", f"{d}/trial_{t}_gtNoCloth.png", f"{d}/trial_{t}_disNoCloth0.png",
           f"results/cloth/imagined/drape_video/trial_{t}_match_0.png", f"results/cloth/imagined/drape_video/trial_{t}_distractor_0.png",
           f"results/cloth/imagined/undrape_video/trial_{t}_target_0.png"]
    ims = [np.array(Image.open(p).convert("RGB").resize((160, 160))) if os.path.exists(p) else np.zeros((160, 160, 3), np.uint8) for p in ims]
    rows.append(np.concatenate(ims, 1))
sheet = Image.fromarray(np.concatenate(rows, 0)); sheet.save("results/cloth/imagined_video_contact_sheet.png"); display(sheet)
print("columns: target | match | distractor | video-draped match | video-draped distractor | video-undraped target")


In [ ]:
#@title 5. Embed stimuli and imagined frames, then analyse
%cd /content/world-maps
args = " ".join(MODELS)
!PYTHONPATH=src python scripts/cloth_extract.py --models {args} 2>&1 | grep -v -i "warn\|Loading weights\|Fetching" | tail -8
!PYTHONPATH=src python scripts/cloth_analyze.py --n-boot 2000
from IPython.display import Image as IM, display
display(IM(filename="results/cloth/figures/trial_correlation_occluded_same.png"))


In [ ]:
#@title 6. Package results for download
%cd /content/world-maps
!zip -q -r /content/world_maps_cloth_video_results.zip results/cloth/cloth_results.json results/cloth/cloth_results.md results/cloth/figures results/cloth/features/models.json results/cloth/imagined_video_contact_sheet.png $(ls results/cloth/imagined/*/*.mp4 2>/dev/null | head -12) 2>/dev/null
!ls -la /content/world_maps_cloth_video_results.zip
from google.colab import files
files.download("/content/world_maps_cloth_video_results.zip")


### If something fails
- **LTX-Video shape errors:** `VIDEO_SIZE` must be a multiple of 32 and `N_FRAMES` = 8k + 1 (49, 57, 65). For Wan use 4k + 1 (49, 81).
- **Out of memory (Wan):** lower `VIDEO_SIZE` to 480 or use `ltx`.
- **Quality check first:** run with `IMAGINE_LIMIT = 12` and look at the contact sheet and mp4s before the full run.
- **Runtime disconnects:** rerun cells 2 and 4; finished clips are skipped.
